# Centroidy terminali i regiony Voronoi

Ten notebook odtwarza endpointy i klastry zgodnie z `trip_endpoint_candidates_map.html`, a potem dzieli obszar na regiony Voronoi.

Wazne: terminale sa liczone tak jak w poprzednim notebooku: `valid_basic & car_like`, `DBSCAN eps=150 m`, `min_samples=8`. Regiony Voronoi sa przycinane do prostokata obroconego o 45 stopni, zeby tworzyly ciagly podzial obszaru zamiast osobnych wysp.


In [2]:
from pathlib import Path
import html

import branca.colormap as cm
import folium
import geopandas as gpd
import numpy as np
import pandas as pd
from scipy.spatial import Voronoi
from shapely import wkt
from shapely.affinity import rotate
from shapely.geometry import MultiPoint, Polygon, box
from sklearn.cluster import DBSCAN
pd.set_option("display.max_columns", 120)

In [3]:
DATA_FILE = Path("data/2024_11_17_17_00_processed.csv")
SEGMENTS_FILE = Path("data/data_2024-11-17.csv")

OUTPUT_CENTROIDS_MAP = Path("terminal_centroids_all_map.html")
OUTPUT_VORONOI_MAP = Path("terminal_voronoi_regions_map.html")
OUTPUT_TERMINALS_GEOJSON = Path("terminal_centroids_all.geojson")
OUTPUT_VORONOI_GEOJSON = Path("terminal_voronoi_regions.geojson")

PARAMS = {
    "time_gap_new_trip_s": 5 * 60,
    "max_plausible_speed_kmh": 180,
    "stop_speed_kmh": 3,
    "long_stop_s": 3 * 60,
    "endpoint_cluster_eps_m": 150,
    "endpoint_cluster_min_samples": 8,
    "min_trip_points": 4,
    "min_trip_duration_s": 30,
    "min_trip_distance_m": 200,
    "bike_max_reported_speed_kmh": 35,
    "bike_max_gps_speed_kmh": 45,
    "bike_median_speed_kmh": 25,
    "car_min_p90_speed_kmh": 45,
    "car_min_max_speed_kmh": 60,
    # Prostokat przyciecia regionow Voronoi. Regiony w takim clipie stykaja sie
    # ze soba i nie robia osobnych wysp.
    "voronoi_clip_rotation_deg": 45,
    "voronoi_clip_padding_m": 600,
}

# Tak jak w trip_endpoint_candidates_map.html: tylko valid_basic i car_like.
USE_BASIC_TRIP_FILTERS = True
USE_CAR_LIKE_FILTER = True

PARAMS

{'time_gap_new_trip_s': 300,
 'max_plausible_speed_kmh': 180,
 'stop_speed_kmh': 3,
 'long_stop_s': 180,
 'endpoint_cluster_eps_m': 150,
 'endpoint_cluster_min_samples': 8,
 'min_trip_points': 4,
 'min_trip_duration_s': 30,
 'min_trip_distance_m': 200,
 'bike_max_reported_speed_kmh': 35,
 'bike_max_gps_speed_kmh': 45,
 'bike_median_speed_kmh': 25,
 'car_min_p90_speed_kmh': 45,
 'car_min_max_speed_kmh': 60,
 'voronoi_clip_rotation_deg': 45,
 'voronoi_clip_padding_m': 600}

In [4]:
df_raw = pd.read_csv(DATA_FILE, parse_dates=["timestamp", "time"])
df_raw = df_raw.sort_values(["vehicle_id", "timestamp"]).copy()

segments_raw = pd.read_csv(SEGMENTS_FILE, parse_dates=["time"])
segments_raw = segments_raw.rename(columns={"segmnet_id": "segment_id"})
segments_daily = (
    segments_raw
    .groupby(["segment_id", "wkt"], as_index=False)
    .agg(
        avg_speed_day=("avg_speed", "mean"),
        median_speed_day=("avg_speed", "median"),
        observations=("avg_speed", "size"),
    )
)
segments_daily["geometry"] = segments_daily["wkt"].apply(wkt.loads)
segments_gdf = gpd.GeoDataFrame(segments_daily, geometry="geometry", crs="EPSG:4326")

print(f"punkty GPS: {len(df_raw):,}")
print(f"segmenty drogowe: {len(segments_gdf):,}")
df_raw.head()

punkty GPS: 81,732
segmenty drogowe: 2,771


,year,month,day,hour,region,vehicle_id,timestamp,time,lon,lat,speed,course,minute,second
20014,2024,11,17,16,25712419,0015a2961cb11f7e79a52504b3597b99bb1d01d6a03bbb...,2024-11-17 16:13:53,2024-11-17 17:13:53,20.95298,52.25468,61,300,13,53
20260,2024,11,17,16,13973275,0015a2961cb11f7e79a52504b3597b99bb1d01d6a03bbb...,2024-11-17 16:14:03,2024-11-17 17:14:03,20.95056,52.25539,64,294,14,3
20508,2024,11,17,16,13973272,0015a2961cb11f7e79a52504b3597b99bb1d01d6a03bbb...,2024-11-17 16:14:13,2024-11-17 17:14:13,20.94872,52.25630,57,338,14,13
20758,2024,11,17,16,24425934,0015a2961cb11f7e79a52504b3597b99bb1d01d6a03bbb...,2024-11-17 16:14:23,2024-11-17 17:14:23,20.94924,52.25768,61,32,14,23
20998,2024,11,17,16,22296730,0015a2961cb11f7e79a52504b3597b99bb1d01d6a03bbb...,2024-11-17 16:14:33,2024-11-17 17:14:33,20.95058,52.25899,61,34,14,33


In [5]:
def haversine_m(lat1, lon1, lat2, lon2):
    radius_m = 6_371_000
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return 2 * radius_m * np.arcsin(np.sqrt(a))


df = df_raw.copy()
df["prev_timestamp"] = df.groupby("vehicle_id")["timestamp"].shift(1)
df["prev_lat"] = df.groupby("vehicle_id")["lat"].shift(1)
df["prev_lon"] = df.groupby("vehicle_id")["lon"].shift(1)

df["time_diff_s"] = (df["timestamp"] - df["prev_timestamp"]).dt.total_seconds()
df["distance_m"] = haversine_m(df["prev_lat"], df["prev_lon"], df["lat"], df["lon"])
df.loc[df["time_diff_s"].isna(), "distance_m"] = np.nan
df["gps_speed_kmh"] = (df["distance_m"] / df["time_diff_s"]) * 3.6

df["is_stationary_point"] = df["speed"].le(PARAMS["stop_speed_kmh"])
df["is_implausible_jump"] = df["gps_speed_kmh"].gt(PARAMS["max_plausible_speed_kmh"])

# Bloki postoju jak w oryginalnym notebooku.
df["stationary_block"] = (
    df.groupby("vehicle_id")["is_stationary_point"]
    .transform(lambda s: s.ne(s.shift()).cumsum())
)
stop_blocks = (
    df[df["is_stationary_point"]]
    .groupby(["vehicle_id", "stationary_block"])
    .agg(
        stop_start=("timestamp", "min"),
        stop_end=("timestamp", "max"),
        stop_points=("timestamp", "size"),
        lat=("lat", "mean"),
        lon=("lon", "mean"),
    )
    .reset_index()
)
stop_blocks["stop_duration_s"] = (stop_blocks["stop_end"] - stop_blocks["stop_start"]).dt.total_seconds()
long_stops = stop_blocks[stop_blocks["stop_duration_s"] >= PARAMS["long_stop_s"]].copy()

long_stop_keys = set(zip(long_stops["vehicle_id"], long_stops["stationary_block"]))
df["is_long_stop_block"] = list(zip(df["vehicle_id"], df["stationary_block"]))
df["is_long_stop_block"] = df["is_long_stop_block"].isin(long_stop_keys)
df["prev_is_long_stop_block"] = df.groupby("vehicle_id")["is_long_stop_block"].shift(1, fill_value=False)

df["new_trip_reason"] = "continue"
df.loc[df["time_diff_s"].isna(), "new_trip_reason"] = "first_point"
df.loc[df["time_diff_s"].gt(PARAMS["time_gap_new_trip_s"]), "new_trip_reason"] = "time_gap"
df.loc[df["is_implausible_jump"], "new_trip_reason"] = "gps_jump"
df.loc[df["prev_is_long_stop_block"] & ~df["is_long_stop_block"], "new_trip_reason"] = "after_long_stop"

df["new_trip"] = df["new_trip_reason"].ne("continue")
df["trip_seq"] = df.groupby("vehicle_id")["new_trip"].cumsum()
df["trip_uid"] = df["vehicle_id"].astype(str) + "_" + df["trip_seq"].astype(str)

df["new_trip_reason"].value_counts()

new_trip_reason
continue           79010
first_point         2424
time_gap             131
gps_jump             109
after_long_stop       58
Name: count, dtype: int64

In [6]:
trip_stats = (
    df.groupby("trip_uid")
    .agg(
        vehicle_id=("vehicle_id", "first"),
        trip_seq=("trip_seq", "first"),
        start_time=("timestamp", "min"),
        end_time=("timestamp", "max"),
        points=("timestamp", "size"),
        start_lat=("lat", "first"),
        start_lon=("lon", "first"),
        end_lat=("lat", "last"),
        end_lon=("lon", "last"),
        origin_region_raw=("region", "first"),
        destination_region_raw=("region", "last"),
        reported_speed_mean=("speed", "mean"),
        reported_speed_median=("speed", "median"),
        reported_speed_p90=("speed", lambda s: s.quantile(0.90)),
        reported_speed_max=("speed", "max"),
        gps_speed_max=("gps_speed_kmh", "max"),
        distance_m=("distance_m", "sum"),
        implausible_jumps=("is_implausible_jump", "sum"),
    )
    .reset_index()
)
trip_stats["duration_s"] = (trip_stats["end_time"] - trip_stats["start_time"]).dt.total_seconds()

trip_stats["too_few_points"] = trip_stats["points"].lt(PARAMS["min_trip_points"])
trip_stats["too_short_duration"] = trip_stats["duration_s"].lt(PARAMS["min_trip_duration_s"])
trip_stats["too_short_distance"] = trip_stats["distance_m"].lt(PARAMS["min_trip_distance_m"])
trip_stats["has_implausible_jump"] = trip_stats["implausible_jumps"].gt(0)
trip_quality_flags = ["too_few_points", "too_short_duration", "too_short_distance", "has_implausible_jump"]
trip_stats["valid_basic"] = ~trip_stats[trip_quality_flags].any(axis=1)

bike_like = (
    trip_stats["reported_speed_max"].le(PARAMS["bike_max_reported_speed_kmh"])
    & trip_stats["gps_speed_max"].le(PARAMS["bike_max_gps_speed_kmh"])
    & trip_stats["reported_speed_median"].le(PARAMS["bike_median_speed_kmh"])
)
car_like = (
    trip_stats["reported_speed_p90"].ge(PARAMS["car_min_p90_speed_kmh"])
    | trip_stats["reported_speed_max"].ge(PARAMS["car_min_max_speed_kmh"])
)
trip_stats["mode_class"] = np.select(
    [bike_like, car_like],
    ["bike_or_slow_candidate", "car_like"],
    default="uncertain_mode",
)

endpoint_source_mask = pd.Series(True, index=trip_stats.index)
if USE_BASIC_TRIP_FILTERS:
    endpoint_source_mask &= trip_stats["valid_basic"]
if USE_CAR_LIKE_FILTER:
    endpoint_source_mask &= trip_stats["mode_class"].eq("car_like")

print(f"wszystkie przejazdy: {len(trip_stats):,}")
print(f"przejazdy uzyte do endpointow: {endpoint_source_mask.sum():,}")
trip_stats.groupby(["valid_basic", "mode_class"]).size().rename("trips").reset_index()

wszystkie przejazdy: 2,722
przejazdy uzyte do endpointow: 2,204


,valid_basic,mode_class,trips
0,False,bike_or_slow_candidate,104
1,False,car_like,227
2,False,uncertain_mode,69
3,True,bike_or_slow_candidate,28
4,True,car_like,2204
5,True,uncertain_mode,90


In [7]:
start_points = trip_stats[endpoint_source_mask][[
    "trip_uid", "vehicle_id", "start_time", "start_lat", "start_lon", "origin_region_raw",
]].rename(columns={
    "start_time": "event_time",
    "start_lat": "lat",
    "start_lon": "lon",
    "origin_region_raw": "source_region",
})
start_points["endpoint_type"] = "origin"

end_points = trip_stats[endpoint_source_mask][[
    "trip_uid", "vehicle_id", "end_time", "end_lat", "end_lon", "destination_region_raw",
]].rename(columns={
    "end_time": "event_time",
    "end_lat": "lat",
    "end_lon": "lon",
    "destination_region_raw": "source_region",
})
end_points["endpoint_type"] = "destination"

endpoints = pd.concat([start_points, end_points], ignore_index=True)
endpoints = endpoints.dropna(subset=["lat", "lon"]).copy()

print(f"endpointy do klastrowania: {len(endpoints):,}")
endpoints.head()

endpointy do klastrowania: 4,408


,trip_uid,vehicle_id,event_time,lat,lon,source_region,endpoint_type
0,0015a2961cb11f7e79a52504b3597b99bb1d01d6a03bbb...,0015a2961cb11f7e79a52504b3597b99bb1d01d6a03bbb...,2024-11-17 16:13:53,52.25468,20.95298,25712419,origin
1,0077e7ad7bffe46df388f4ac582d8e875f937d5b486bc3...,0077e7ad7bffe46df388f4ac582d8e875f937d5b486bc3...,2024-11-17 16:34:09,52.26997,20.99809,21087632,origin
2,00ac440e057201225e7db09525027138759d952e787277...,00ac440e057201225e7db09525027138759d952e787277...,2024-11-17 16:34:12,52.30655,21.06170,20766209,origin
3,00ac440e057201225e7db09525027138759d952e787277...,00ac440e057201225e7db09525027138759d952e787277...,2024-11-17 16:49:30,52.30512,21.05664,9608649,origin
4,00c5c59bede3256a7a16512887c7e9f8ae0cf83cbbf6c6...,00c5c59bede3256a7a16512887c7e9f8ae0cf83cbbf6c6...,2024-11-17 16:36:12,52.29961,21.03551,25791822,origin


In [8]:
earth_radius_m = 6_371_000
eps_rad = PARAMS["endpoint_cluster_eps_m"] / earth_radius_m
coords_rad = np.radians(endpoints[["lat", "lon"]].to_numpy())

clusterer = DBSCAN(
    eps=eps_rad,
    min_samples=PARAMS["endpoint_cluster_min_samples"],
    metric="haversine",
)
endpoints["endpoint_cluster"] = clusterer.fit_predict(coords_rad)

print("endpointy uzyte do DBSCAN:", len(endpoints))
print("klastry jak w trip_endpoint_candidates_map:", int(endpoints["endpoint_cluster"].ge(0).groupby(endpoints["endpoint_cluster"]).any().loc[lambda s: s.index >= 0].sum()))
print("noise endpoints:", int(endpoints["endpoint_cluster"].eq(-1).sum()))
endpoints["endpoint_cluster"].value_counts().sort_index().head()


endpointy uzyte do DBSCAN: 4408
klastry jak w trip_endpoint_candidates_map: 48
noise endpoints: 129


endpoint_cluster
-1    129
 0    398
 1    397
 2     32
 3    113
Name: count, dtype: int64

In [9]:
cluster_stats = (
    endpoints[endpoints["endpoint_cluster"] >= 0]
    .groupby("endpoint_cluster")
    .agg(
        lat=("lat", "mean"),
        lon=("lon", "mean"),
        events=("trip_uid", "size"),
        origins=("endpoint_type", lambda s: (s == "origin").sum()),
        destinations=("endpoint_type", lambda s: (s == "destination").sum()),
        source_regions=("source_region", "nunique"),
    )
    .reset_index()
    .sort_values("events", ascending=False)
)
cluster_stats["terminal_id"] = "cluster_" + cluster_stats["endpoint_cluster"].astype(str)
cluster_stats["terminal_name"] = cluster_stats["terminal_id"]

terminal_gdf = gpd.GeoDataFrame(
    cluster_stats,
    geometry=gpd.points_from_xy(cluster_stats["lon"], cluster_stats["lat"]),
    crs="EPSG:4326",
)
terminal_gdf.to_file(OUTPUT_TERMINALS_GEOJSON, driver="GeoJSON")

print(f"terminale/centroidy jak w trip_endpoint_candidates_map: {len(terminal_gdf):,}")
cluster_stats.head(30)


terminale/centroidy jak w trip_endpoint_candidates_map: 48


,endpoint_cluster,lat,lon,events,origins,destinations,source_regions,terminal_id,terminal_name
7,7,52.217603,20.865222,721,377,344,43,cluster_7,cluster_7
5,5,52.307646,21.095602,627,339,288,26,cluster_5,cluster_5
10,10,52.291815,20.973781,467,233,234,19,cluster_10,cluster_10
0,0,52.255466,20.952587,398,212,186,47,cluster_0,cluster_0
1,1,52.269707,20.998854,397,191,206,21,cluster_1,cluster_1
27,27,52.303708,20.988934,250,120,130,18,cluster_27,cluster_27
14,14,52.280587,21.010720,143,79,64,12,cluster_14,cluster_14
3,3,52.297974,21.031511,113,59,54,47,cluster_3,cluster_3
23,23,52.239424,20.896751,80,29,51,19,cluster_23,cluster_23
17,17,52.307148,21.080001,67,25,42,24,cluster_17,cluster_17


In [10]:
m = folium.Map(location=[df["lat"].mean(), df["lon"].mean()], zoom_start=12, tiles="CartoDB positron")

# Kontekst drogowy.
for _, row in segments_gdf.iterrows():
    coords = [(lat, lon) for lon, lat in row.geometry.coords]
    folium.PolyLine(coords, color="#808080", weight=1, opacity=0.25).add_to(m)

# Lekka probka endpointow, zeby widziec chmure bez zabicia mapy.
sample_endpoints = endpoints.sample(min(len(endpoints), 8000), random_state=42)
endpoint_colors = {"origin": "#1f77b4", "destination": "#d62728"}
for _, row in sample_endpoints.iterrows():
    folium.CircleMarker(
        [row["lat"], row["lon"]],
        radius=2,
        color=endpoint_colors.get(row["endpoint_type"], "#666"),
        fill=True,
        fill_opacity=0.25,
        opacity=0.25,
        tooltip=f"{row.endpoint_type}<br>cluster: {row.endpoint_cluster}<br>trip: {row.trip_uid}",
    ).add_to(m)

# Wszystkie centroidy, bez top-N i bez progu minimalnego rozmiaru.
for _, row in cluster_stats.iterrows():
    radius = 4 + min(12, np.log1p(row["events"]) * 1.8)
    folium.CircleMarker(
        [row["lat"], row["lon"]],
        radius=radius,
        color="#111111",
        fill=True,
        fill_color="#ffd23f",
        fill_opacity=0.9,
        weight=1,
        tooltip=(
            f"{html.escape(row.terminal_id)}<br>"
            f"events: {row.events}<br>"
            f"origins: {row.origins}<br>"
            f"destinations: {row.destinations}<br>"
            f"source regions: {row.source_regions}"
        ),
    ).add_to(m)

legend_html = (
    '<div style="position: fixed; bottom: 30px; left: 30px; z-index: 9999; background: white; padding: 10px 12px; border: 1px solid #999; border-radius: 4px; font-size: 13px;">'
    '<b>Centroidy terminali</b><br>'
    '<span style="color:#1f77b4;">&bull;</span> endpoint origin, probka<br>'
    '<span style="color:#d62728;">&bull;</span> endpoint destination, probka<br>'
    '<span style="color:#111; background:#ffd23f;">&bull;</span> centroid klastra z trip_endpoint_candidates_map<br>'
    'DBSCAN min_samples = 8, valid_basic & car_like</div>'
)
m.get_root().html.add_child(folium.Element(legend_html))

m.save(OUTPUT_CENTROIDS_MAP)
OUTPUT_CENTROIDS_MAP

PosixPath('terminal_centroids_all_map.html')

In [11]:
def voronoi_finite_polygons_2d(vor, radius=None):
    # Adaptacja popularnej funkcji do domykania nieskonczonych komorek Voronoi 2D.
    if vor.points.shape[1] != 2:
        raise ValueError("Voronoi wymaga punktow 2D")

    new_regions = []
    new_vertices = vor.vertices.tolist()
    center = vor.points.mean(axis=0)
    if radius is None:
        radius = np.ptp(vor.points, axis=0).max() * 2

    all_ridges = {}
    for (p1, p2), (v1, v2) in zip(vor.ridge_points, vor.ridge_vertices):
        all_ridges.setdefault(p1, []).append((p2, v1, v2))
        all_ridges.setdefault(p2, []).append((p1, v1, v2))

    for p1, region_idx in enumerate(vor.point_region):
        vertices = vor.regions[region_idx]
        if all(v >= 0 for v in vertices):
            new_regions.append(vertices)
            continue

        ridges = all_ridges[p1]
        new_region = [v for v in vertices if v >= 0]

        for p2, v1, v2 in ridges:
            if v2 < 0:
                v1, v2 = v2, v1
            if v1 >= 0:
                continue

            tangent = vor.points[p2] - vor.points[p1]
            tangent /= np.linalg.norm(tangent)
            normal = np.array([-tangent[1], tangent[0]])
            midpoint = vor.points[[p1, p2]].mean(axis=0)
            direction = np.sign(np.dot(midpoint - center, normal)) * normal
            far_point = vor.vertices[v2] + direction * radius
            new_region.append(len(new_vertices))
            new_vertices.append(far_point.tolist())

        vertices_arr = np.asarray([new_vertices[v] for v in new_region])
        centroid = vertices_arr.mean(axis=0)
        angles = np.arctan2(vertices_arr[:, 1] - centroid[1], vertices_arr[:, 0] - centroid[0])
        new_region = [v for _, v in sorted(zip(angles, new_region))]
        new_regions.append(new_region)

    return new_regions, np.asarray(new_vertices)

## Regiony Voronoi w obroconym prostokacie

Zamiast buforow wokol terminali, ktore tworzyly nie-stykajace sie wyspy, uzywamy jednego prostokata przyciecia. Prostokat jest liczony w ukladzie EPSG:2180, obracany o `voronoi_clip_rotation_deg` stopni i powiekszany o `voronoi_clip_padding_m`. W takim obszarze komorki Voronoi dziela caly prostokat i stykaja sie granicami.


In [12]:
terminals_2180 = terminal_gdf.to_crs(epsg=2180).copy()
segments_2180 = segments_gdf.to_crs(epsg=2180)

if len(terminals_2180) < 3:
    raise ValueError("Voronoi wymaga co najmniej trzech terminali dla tej implementacji")

points_xy = np.column_stack([terminals_2180.geometry.x, terminals_2180.geometry.y])

# Budujemy jeden prostokat przyciecia, obrocony o zadany kat.
# Uzywamy unary_union zamiast MultiPoint, bo segments_2180 zawiera LineStringi.
angle = PARAMS["voronoi_clip_rotation_deg"]
padding = PARAMS["voronoi_clip_padding_m"]

clip_source_geom = terminals_2180.geometry.unary_union.union(
    segments_2180.geometry.unary_union
)

center = clip_source_geom.centroid

rotated_source = rotate(clip_source_geom, -angle, origin=center)
minx, miny, maxx, maxy = rotated_source.bounds

rotated_clip_rect = box(
    minx - padding,
    miny - padding,
    maxx + padding,
    maxy + padding,
)

clip_geom = rotate(rotated_clip_rect, angle, origin=center)

clip_gdf = gpd.GeoDataFrame(
    [{
        "name": "rotated_voronoi_clip",
        "rotation_deg": angle,
        "padding_m": padding,
    }],
    geometry=[clip_geom],
    crs="EPSG:2180",
)

vor = Voronoi(points_xy)
regions, vertices = voronoi_finite_polygons_2d(vor, radius=100_000)

polygons = []

for region in regions:
    polygon = Polygon(vertices[region])

    # Naprawa potencjalnie niepoprawnych geometrii.
    if not polygon.is_valid:
        polygon = polygon.buffer(0)

    clipped = polygon.intersection(clip_geom)

    if not clipped.is_empty:
        polygons.append(clipped)
    else:
        polygons.append(None)

voronoi_gdf = terminals_2180.drop(columns="geometry").copy()
voronoi_gdf["geometry"] = polygons

voronoi_gdf = gpd.GeoDataFrame(
    voronoi_gdf,
    geometry="geometry",
    crs="EPSG:2180",
)

voronoi_gdf = voronoi_gdf.dropna(subset=["geometry"])
voronoi_gdf = voronoi_gdf[~voronoi_gdf.geometry.is_empty].copy()

voronoi_gdf["area_km2"] = voronoi_gdf.geometry.area / 1_000_000

voronoi_gdf_4326 = voronoi_gdf.to_crs(epsg=4326)
clip_gdf_4326 = clip_gdf.to_crs(epsg=4326)

voronoi_gdf_4326.to_file(
    OUTPUT_VORONOI_GEOJSON,
    driver="GeoJSON",
)

print(f"regiony Voronoi: {len(voronoi_gdf_4326):,}")
print("clip rotation deg:", angle)
print("clip padding m:", padding)

voronoi_gdf_4326[
    ["terminal_id", "events", "area_km2"]
].sort_values(
    "events",
    ascending=False,
).head(20)


regiony Voronoi: 48
clip rotation deg: 45
clip padding m: 600


/tmp/ipykernel_11444/3050589047.py:14: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  clip_source_geom = terminals_2180.geometry.unary_union.union(
/tmp/ipykernel_11444/3050589047.py:15: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  segments_2180.geometry.unary_union


,terminal_id,events,area_km2
7,cluster_7,721,14.159494
5,cluster_5,627,3.180653
10,cluster_10,467,2.986335
0,cluster_0,398,4.140815
1,cluster_1,397,7.463072
27,cluster_27,250,3.754077
14,cluster_14,143,2.306364
3,cluster_3,113,3.130617
23,cluster_23,80,5.548338
17,cluster_17,67,2.672273


In [13]:
region_colormap = cm.LinearColormap(
    ["#e8f4f8", "#a8ddb5", "#43a2ca", "#0868ac"],
    vmin=cluster_stats["events"].min(),
    vmax=cluster_stats["events"].max(),
)

region_colormap.caption = "Liczba endpointow w terminalu"

m_v = folium.Map(
    location=[df["lat"].mean(), df["lon"].mean()],
    zoom_start=12,
    tiles="CartoDB positron",
)

folium.GeoJson(
    voronoi_gdf_4326,
    name="Regiony Voronoi terminali",
    style_function=lambda feature: {
        "fillColor": region_colormap(feature["properties"]["events"]),
        "color": "#333333",
        "weight": 1,
        "fillOpacity": 0.35,
    },
    tooltip=folium.GeoJsonTooltip(
        fields=[
            "terminal_id",
            "events",
            "origins",
            "destinations",
            "area_km2",
        ],
        aliases=[
            "terminal",
            "endpointy",
            "origin",
            "destination",
            "area km2",
        ],
        localize=True,
    ),
).add_to(m_v)

folium.GeoJson(
    clip_gdf_4326,
    name="Obrocony prostokat przyciecia",
    style_function=lambda feature: {
        "fillColor": None,
        "color": "#111111",
        "weight": 2,
        "dashArray": "6 4",
        "fillOpacity": 0,
    },
).add_to(m_v)

for _, row in segments_gdf.iterrows():
    coords = [(lat, lon) for lon, lat in row.geometry.coords]

    folium.PolyLine(
        coords,
        color="#555555",
        weight=1,
        opacity=0.35,
    ).add_to(m_v)

for _, row in cluster_stats.iterrows():
    folium.CircleMarker(
        [row["lat"], row["lon"]],
        radius=4 + min(12, np.log1p(row["events"]) * 1.8),
        color="#111111",
        fill=True,
        fill_color="#ffd23f",
        fill_opacity=0.95,
        weight=1,
        tooltip=(
            f"{html.escape(row.terminal_id)}<br>"
            f"events: {row.events}<br>"
            f"origins: {row.origins}<br>"
            f"destinations: {row.destinations}"
        ),
    ).add_to(m_v)

region_colormap.add_to(m_v)

folium.LayerControl().add_to(m_v)

m_v.save(OUTPUT_VORONOI_MAP)

OUTPUT_VORONOI_MAP


PosixPath('terminal_voronoi_regions_map.html')

In [14]:
# Tabela pomocnicza: terminale posortowane po liczbie endpointow.
cluster_stats[[
    "terminal_id", "endpoint_cluster", "lat", "lon", "events", "origins", "destinations", "source_regions",
]].sort_values("events", ascending=False).reset_index(drop=True)

,terminal_id,endpoint_cluster,lat,lon,events,origins,destinations,source_regions
0,cluster_7,7,52.217603,20.865222,721,377,344,43
1,cluster_5,5,52.307646,21.095602,627,339,288,26
2,cluster_10,10,52.291815,20.973781,467,233,234,19
3,cluster_0,0,52.255466,20.952587,398,212,186,47
4,cluster_1,1,52.269707,20.998854,397,191,206,21
5,cluster_27,27,52.303708,20.988934,250,120,130,18
6,cluster_14,14,52.280587,21.010720,143,79,64,12
7,cluster_3,3,52.297974,21.031511,113,59,54,47
8,cluster_23,23,52.239424,20.896751,80,29,51,19
9,cluster_17,17,52.307148,21.080001,67,25,42,24


In [26]:
active = cluster_stats["endpoint_cluster"][:25]


# Dynamic terminal network recomputation

This section adds:
- active/inactive terminal management
- dynamic Voronoi recomputation
- terminal removal/restoration
- endpoint reassignment
- dynamic OD matrix recomputation
- interactive scenario analysis


In [27]:

ACTIVE_TERMINALS = set(active)
REMOVED_TERMINALS = set()

print("active terminals:", len(ACTIVE_TERMINALS))


active terminals: 25


In [28]:

def remove_terminal(terminal_id):

    global ACTIVE_TERMINALS
    global REMOVED_TERMINALS

    if terminal_id not in ACTIVE_TERMINALS:
        print(f"terminal {terminal_id} already removed")
        return

    ACTIVE_TERMINALS.remove(terminal_id)
    REMOVED_TERMINALS.add(terminal_id)

    print(f"removed terminal: {terminal_id}")
    print(f"remaining terminals: {len(ACTIVE_TERMINALS)}")


In [29]:

def restore_terminal(terminal_id):

    global ACTIVE_TERMINALS
    global REMOVED_TERMINALS

    if terminal_id in REMOVED_TERMINALS:

        REMOVED_TERMINALS.remove(terminal_id)
        ACTIVE_TERMINALS.add(terminal_id)

        print(f"restored terminal: {terminal_id}")


In [30]:
def get_active_terminals():

    return cluster_stats[
        cluster_stats["endpoint_cluster"].isin(ACTIVE_TERMINALS)
    ].copy()



## Dynamic Voronoi recomputation


In [31]:

def rebuild_voronoi():

    active_terminals = get_active_terminals()

    terminals_gdf_dynamic = gpd.GeoDataFrame(
        active_terminals,
        geometry=gpd.points_from_xy(
            active_terminals.lon,
            active_terminals.lat,
        ),
        crs="EPSG:4326",
    )

    terminals_2180 = terminals_gdf_dynamic.to_crs(epsg=2180)

    if len(terminals_2180) < 3:
        raise ValueError("Voronoi requires at least 3 active terminals")

    points_xy = np.column_stack([
        terminals_2180.geometry.x,
        terminals_2180.geometry.y,
    ])

    vor = Voronoi(points_xy)

    regions, vertices = voronoi_finite_polygons_2d(
        vor,
        radius=100_000,
    )

    polygons = []

    for region in regions:

        polygon = Polygon(vertices[region])

        if not polygon.is_valid:
            polygon = polygon.buffer(0)

        clipped = polygon.intersection(clip_geom)

        polygons.append(clipped)

    voronoi_dynamic = terminals_2180.copy()

    voronoi_dynamic["geometry"] = polygons

    voronoi_dynamic = gpd.GeoDataFrame(
        voronoi_dynamic,
        geometry="geometry",
        crs="EPSG:2180",
    )

    voronoi_dynamic = voronoi_dynamic[
        ~voronoi_dynamic.geometry.is_empty
    ].copy()

    voronoi_dynamic["area_km2"] = (
        voronoi_dynamic.geometry.area / 1_000_000
    )

    return voronoi_dynamic.to_crs(epsg=4326)



## Dynamic map plotting


In [32]:

def plot_dynamic_voronoi(voronoi_dynamic):

    m = folium.Map(
        location=[df["lat"].mean(), df["lon"].mean()],
        zoom_start=12,
        tiles="CartoDB positron",
    )

    folium.GeoJson(
        voronoi_dynamic,
        name="Dynamic Voronoi",
        style_function=lambda feature: {
            "fillColor": "#66c2a5",
            "color": "#222222",
            "weight": 1,
            "fillOpacity": 0.35,
        },
        tooltip=folium.GeoJsonTooltip(
            fields=[
                "terminal_id",
                "events",
                "origins",
                "destinations",
                "area_km2",
            ],
            aliases=[
                "terminal",
                "events",
                "origins",
                "destinations",
                "area km2",
            ],
        ),
    ).add_to(m)

    for _, row in voronoi_dynamic.iterrows():

        folium.CircleMarker(
            [row["lat"], row["lon"]],
            radius=5 + min(10, np.log1p(row["events"])),
            color="#111111",
            fill=True,
            fill_color="#ffd23f",
            fill_opacity=0.9,
            weight=1,
            tooltip=f"terminal: {row['terminal_id']}",
        ).add_to(m)

    folium.LayerControl().add_to(m)

    return m



## Endpoint reassignment

This stage reassigns trip endpoints to the currently active Voronoi regions.


In [33]:

def assign_points_to_terminals(points_gdf, voronoi_dynamic):

    joined = gpd.sjoin(
        points_gdf,
        voronoi_dynamic[
            ["terminal_id", "geometry"]
        ],
        how="left",
        predicate="within",
    )

    return joined.drop(columns=["index_right"])



## Dynamic OD matrix recomputation


In [34]:

def build_dynamic_od_matrix(
    trips_df,
    origin_assignment,
    destination_assignment,
):

    tmp = trips_df.copy()

    tmp["origin_terminal"] = origin_assignment["terminal_id"].values
    tmp["destination_terminal"] = destination_assignment["terminal_id"].values

    od_matrix = (
        tmp
        .groupby([
            "origin_terminal",
            "destination_terminal",
        ])
        .size()
        .reset_index(name="trip_count")
    )

    return od_matrix



## Full recomputation pipeline


In [35]:

def recompute_network():

    voronoi_dynamic = rebuild_voronoi()

    print("active terminals:", len(voronoi_dynamic))

    return voronoi_dynamic



## Example scenario analysis


In [37]:

# Example:
# remove_terminal(17)
voronoi_dynamic = recompute_network()

plot_dynamic_voronoi(voronoi_dynamic)


active terminals: 25



## Recommended next steps

1. Add endpoint GeoDataFrames for origins and destinations
2. Reassign all trip endpoints after terminal removal
3. Recompute OD matrix automatically
4. Add merge/split terminal operations
5. Add scenario comparison metrics
6. Add network resilience analysis
